# AgroLens — Potato Disease Classifier Training Notebook

Trains a MobileNetV2-based leaf disease classifier on the PlantVillage dataset (potato
classes), evaluates it, saves it, and includes a real-time image-upload inference tester.

This version already includes the accuracy fixes found while training on tomato earlier:
`alpha=0.75` (more capacity than the minimum 0.35) and a gentler fine-tuning pass
(fewer unfrozen layers, lower learning rate) to avoid the training instability seen before.

**How to use this notebook:**
1. Download the PlantVillage dataset from Kaggle (abdallahalidev/plantvillage-dataset) and unzip it.
2. In the **Configuration** cell below, set `DATASET_DIR` to the unzipped folder path.
3. Run all cells top to bottom (`Kernel > Restart & Run All`).
4. Use the last section to upload any leaf photo and see a live prediction.

This notebook trains on your **laptop/Colab GPU**, not on the ESP32-CAM/Pi. It only runs the
final quantized model on the hardware — that comes later, in the deployment stage.


## 0. Install & import dependencies

In [ ]:
# Run this once. Restart the kernel if TensorFlow was just installed.
# !pip install tensorflow scikit-learn matplotlib seaborn pillow ipywidgets --quiet


In [ ]:
import os
import shutil
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))


## 1. Configuration — EDIT THIS CELL

This notebook is configured for the **abdallahalidev/plantvillage-dataset** Kaggle dataset
(38 classes, ~54,000 images: `https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset`).

After downloading and unzipping, this dataset's folder structure looks like:

```
PlantVillage/
    color/
        Potato___Early_blight/
        Potato___Late_blight/
        Potato___healthy/
        Tomato___Bacterial_spot/
        ...  (38 class folders total, triple-underscore names)
    grayscale/
        ... (same classes, grayscale versions — not used here)
    segmented/
        ... (same classes, background-removed versions — not used here)
```

Set `DATASET_DIR` to the **outer** unzipped folder (the one containing `color/`,
`grayscale/`, `segmented/`) — the cell below automatically finds and uses the `color/`
subfolder, since that's closest to what the camera will actually capture in the field.

`FILTER_KEYWORD = "Potato"` keeps only the 3 potato classes (Early blight, Late blight,
healthy). Set it to `None` to use every class in the folder instead.


In [ ]:
# ============================================================
# EDIT THESE VALUES
# ============================================================

DATASET_DIR = "/path/to/your/PlantVillage"   # <-- CHANGE THIS to your unzipped dataset folder
                                               # (the outer folder containing color/grayscale/segmented,
                                               # or the folder containing class subfolders directly)
                                               # On Windows, use a raw string: r"C:\path\to\dataset"

FILTER_KEYWORD = "Potato"   # keep only classes whose folder name contains this string.
                             # Set to None to use ALL classes in DATASET_DIR.

IMG_SIZE = (96, 96)         # 96x96 keeps the model small enough for later TFLite Micro conversion.
                             # Use 160x160 or 224x224 only if you plan to prune/quantize aggressively.

BATCH_SIZE = 32
EPOCHS = 15
SEED = 42

WORK_DIR = "./agrolens_potato_work"        # scratch folder this notebook creates
FILTERED_DIR = os.path.join(WORK_DIR, "filtered_dataset")
SPLIT_DIR = os.path.join(WORK_DIR, "split_dataset")
MODEL_SAVE_PATH = os.path.join(WORK_DIR, "agrolens_potato_model.keras")
MODEL_SAVEDMODEL_PATH = os.path.join(WORK_DIR, "agrolens_potato_savedmodel")

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

os.makedirs(WORK_DIR, exist_ok=True)
print("Config set. Dataset dir:", DATASET_DIR)


## 2. Filter to selected classes (optional)

First auto-detects whether `DATASET_DIR` is the outer folder (containing `color/`,
`grayscale/`, `segmented/`) or already the folder of class subfolders directly. Then, if
`FILTER_KEYWORD` is set, copies only the matching class folders into a smaller working
directory — keeps training fast and keeps the final model small enough for the target
hardware's memory budget.


In [ ]:
assert os.path.isdir(DATASET_DIR), f"DATASET_DIR does not exist: {DATASET_DIR}. Fix the path in the config cell above."

# Auto-detect the abdallahalidev color/grayscale/segmented layout
if os.path.isdir(os.path.join(DATASET_DIR, "color")):
    SOURCE_DIR = os.path.join(DATASET_DIR, "color")
    print("Detected color/grayscale/segmented layout — using the 'color' subfolder.")
else:
    SOURCE_DIR = DATASET_DIR
    print("Using DATASET_DIR directly as the class-folder root.")

all_classes = sorted([d for d in os.listdir(SOURCE_DIR) if os.path.isdir(os.path.join(SOURCE_DIR, d))])
print(f"Found {len(all_classes)} classes in dataset folder.")

if FILTER_KEYWORD:
    selected_classes = [c for c in all_classes if FILTER_KEYWORD.lower() in c.lower()]
else:
    selected_classes = all_classes

assert len(selected_classes) > 0, "No classes matched your FILTER_KEYWORD. Check spelling or set it to None."

print(f"Using {len(selected_classes)} classes:")
for c in selected_classes:
    print("  -", c)

if os.path.exists(FILTERED_DIR):
    shutil.rmtree(FILTERED_DIR)
os.makedirs(FILTERED_DIR, exist_ok=True)

for c in selected_classes:
    src = os.path.join(SOURCE_DIR, c)
    dst = os.path.join(FILTERED_DIR, c)
    shutil.copytree(src, dst)

print("Filtered dataset ready at:", FILTERED_DIR)


## 3. Clean corrupt / unreadable images

In [ ]:
from PIL import Image

def clean_dataset(root_dir):
    removed = 0
    for subdir, _, files in os.walk(root_dir):
        for f in files:
            path = os.path.join(subdir, f)
            try:
                img = Image.open(path)
                img.verify()
            except Exception:
                os.remove(path)
                removed += 1
    return removed

removed_count = clean_dataset(FILTERED_DIR)
print(f"Removed {removed_count} corrupt/unreadable files.")


## 4. Train / Validation / Test split

Stratified 70/15/15 split so every class is proportionally represented in each split.


In [ ]:
def split_dataset(source_dir, dest_dir, split_ratio=(0.7, 0.15, 0.15), seed=SEED):
    assert abs(sum(split_ratio) - 1.0) < 1e-6
    if os.path.exists(dest_dir):
        shutil.rmtree(dest_dir)

    classes = sorted([d for d in os.listdir(source_dir) if os.path.isdir(os.path.join(source_dir, d))])
    rng = random.Random(seed)

    for split_name in ["train", "val", "test"]:
        for c in classes:
            os.makedirs(os.path.join(dest_dir, split_name, c), exist_ok=True)

    for c in classes:
        files = os.listdir(os.path.join(source_dir, c))
        rng.shuffle(files)
        n = len(files)
        n_train = int(n * split_ratio[0])
        n_val = int(n * split_ratio[1])

        train_files = files[:n_train]
        val_files = files[n_train:n_train + n_val]
        test_files = files[n_train + n_val:]

        for f in train_files:
            shutil.copy(os.path.join(source_dir, c, f), os.path.join(dest_dir, "train", c, f))
        for f in val_files:
            shutil.copy(os.path.join(source_dir, c, f), os.path.join(dest_dir, "val", c, f))
        for f in test_files:
            shutil.copy(os.path.join(source_dir, c, f), os.path.join(dest_dir, "test", c, f))

    return classes

class_names = split_dataset(FILTERED_DIR, SPLIT_DIR)
print("Classes:", class_names)

for split_name in ["train", "val", "test"]:
    total = sum(len(os.listdir(os.path.join(SPLIT_DIR, split_name, c))) for c in class_names)
    print(f"{split_name}: {total} images")


## 5. Load data + augmentation

Augmentation is applied only to the training set — it simulates the rotation, lighting,
and noise variation the camera will see in the field. Validation and test sets stay
unaugmented so evaluation numbers are honest.


In [ ]:
train_dir = os.path.join(SPLIT_DIR, "train")
val_dir = os.path.join(SPLIT_DIR, "val")
test_dir = os.path.join(SPLIT_DIR, "test")

train_ds_raw = tf.keras.utils.image_dataset_from_directory(
    train_dir, image_size=IMG_SIZE, batch_size=BATCH_SIZE, seed=SEED, label_mode="categorical"
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    val_dir, image_size=IMG_SIZE, batch_size=BATCH_SIZE, seed=SEED, label_mode="categorical"
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir, image_size=IMG_SIZE, batch_size=BATCH_SIZE, seed=SEED, label_mode="categorical", shuffle=False
)

class_names = train_ds_raw.class_names
num_classes = len(class_names)
print("Class names (index order used by the model):", class_names)

data_augmentation = tf.keras.Sequential([
    layers.RandomRotation(0.06),       # ~20 degrees
    layers.RandomBrightness(0.15),
    layers.RandomContrast(0.15),
    layers.RandomFlip("horizontal"),
    layers.RandomZoom(0.1),
])

normalization_layer = layers.Rescaling(1./255)

def prep_train(ds):
    ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.map(lambda x, y: (normalization_layer(x), y), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

def prep_eval(ds):
    ds = ds.map(lambda x, y: (normalization_layer(x), y), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

train_ds = prep_train(train_ds_raw)
val_ds = prep_eval(val_ds)
test_ds = prep_eval(test_ds)


In [ ]:
# Quick sanity check: visualize a batch of augmented training images
plt.figure(figsize=(10, 10))
for images, labels in train_ds_raw.take(1):
    aug_images = data_augmentation(images, training=True)
    for i in range(min(9, images.shape[0])):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(aug_images[i].numpy().astype("uint8"))
        plt.title(class_names[int(tf.argmax(labels[i]))], fontsize=9)
        plt.axis("off")
plt.tight_layout()
plt.show()


## 6. Model architecture — MobileNetV2 transfer learning

`alpha=0.75` (not the minimum 0.35) is used here for more classification capacity — this
was the fix that helped with the tomato run's low recall on visually similar disease
classes. You can shrink it back down during hardware-specific quantization later if the
final model doesn't fit in memory.


In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights="imagenet",
    alpha=0.75,
)
base_model.trainable = False   # freeze backbone for initial training

model = models.Sequential([
    layers.Input(shape=IMG_SIZE + (3,)),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(num_classes, activation="softmax"),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()


In [ ]:
# Compute class weights to counteract class imbalance (e.g. potato_healthy has far
# fewer images than the disease classes in PlantVillage) — without this, the model tends
# to under-predict the rare class.
from sklearn.utils.class_weight import compute_class_weight

train_labels = []
for c_idx, c_name in enumerate(class_names):
    n_files = len(os.listdir(os.path.join(train_dir, c_name)))
    train_labels.extend([c_idx] * n_files)

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(class_names)),
    y=train_labels
)
class_weight_dict = dict(enumerate(class_weights_array))
print("Class weights:", dict(zip(class_names, class_weights_array)))


## 7. Training

Trains the classification head first (backbone frozen), then optionally fine-tunes a small
number of the backbone's last layers at a very low learning rate. Only 10 layers are
unfrozen here (not 20) — unfreezing too much caused a training-instability loss spike on
an earlier run. `class_weight_dict` (computed above) is passed into both training calls to
counteract class imbalance.


In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    class_weight=class_weight_dict,
)


In [ ]:
# Optional gentle fine-tuning pass. Skip this cell entirely if val accuracy is already
# good enough after the frozen-backbone training above.

base_model.trainable = True
fine_tune_at = len(base_model.layers) - 10   # fewer unfrozen layers avoids training instability
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-6),   # lower LR avoids catastrophic forgetting
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

fine_tune_epochs = 5
history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=fine_tune_epochs,
    callbacks=callbacks,
    class_weight=class_weight_dict,
)


In [ ]:
# Plot training curves
def plot_history(h1, h2=None):
    acc = h1.history["accuracy"]
    val_acc = h1.history["val_accuracy"]
    loss = h1.history["loss"]
    val_loss = h1.history["val_loss"]

    if h2 is not None:
        acc = acc + h2.history["accuracy"]
        val_acc = val_acc + h2.history["val_accuracy"]
        loss = loss + h2.history["loss"]
        val_loss = val_loss + h2.history["val_loss"]

    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(acc, label="train")
    plt.plot(val_acc, label="val")
    plt.title("Accuracy")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(loss, label="train")
    plt.plot(val_loss, label="val")
    plt.title("Loss")
    plt.legend()
    plt.tight_layout()
    plt.show()

if "history" not in dir():
    print("No training history found yet. Run the 'model.fit(...)' training cell above first, "
          "then re-run this cell.")
else:
    plot_history(history, history_fine if "history_fine" in dir() else None)


## 8. Evaluation on the held-out test set

Reports accuracy, precision, recall, F1-score (per class and macro-averaged), and a
confusion matrix.


In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print(f"Test accuracy: {test_acc:.4f}")
print(f"Test loss: {test_loss:.4f}")

y_true = []
y_pred = []
for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print("\nClassification report:")
print(classification_report(y_true, y_pred, target_names=class_names))

precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro")
print(f"Macro Precision: {precision:.4f}  Macro Recall: {recall:.4f}  Macro F1: {f1:.4f}")


In [ ]:
# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion matrix — test set")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## 9. Save the trained model

Saves both a `.keras` file (for reuse in this notebook / further training) and a
TensorFlow SavedModel folder (needed later for TFLite / TFLite Micro conversion in the
next stage of the project).


In [ ]:
model.save(MODEL_SAVE_PATH)
model.export(MODEL_SAVEDMODEL_PATH)

with open(os.path.join(WORK_DIR, "class_names.txt"), "w") as f:
    for c in class_names:
        f.write(c + "\n")

print("Saved model to:", MODEL_SAVE_PATH)
print("Saved SavedModel export to:", MODEL_SAVEDMODEL_PATH)
print("Saved class names to:", os.path.join(WORK_DIR, "class_names.txt"))
print("\nNext step (separate notebook/section): quantization + TFLite/TFLite Micro conversion for your target hardware.")


## 10. Real-time inference — upload an image and get a live prediction

Run the cell below, click "Upload", pick any potato leaf photo from your computer, and the
model's prediction (disease name + confidence) will display immediately below it.


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image
import io

uploader = widgets.FileUpload(accept="image/*", multiple=False, description="Upload leaf photo")
output = widgets.Output()

def predict_image(img: Image.Image):
    img_resized = img.convert("RGB").resize(IMG_SIZE)
    arr = np.array(img_resized).astype("float32") / 255.0
    arr = np.expand_dims(arr, axis=0)
    preds = model.predict(arr, verbose=0)[0]
    top_idx = int(np.argmax(preds))
    return class_names[top_idx], float(preds[top_idx]), preds

def on_upload_change(change):
    with output:
        clear_output()
        if not uploader.value:
            return
        uploaded_file = list(uploader.value.values())[0] if isinstance(uploader.value, dict) else uploader.value[0]
        content = uploaded_file["content"] if isinstance(uploaded_file, dict) else uploaded_file.content
        img = Image.open(io.BytesIO(content))

        predicted_class, confidence, all_preds = predict_image(img)

        plt.figure(figsize=(4, 4))
        plt.imshow(img)
        plt.axis("off")
        plt.title(f"{predicted_class}\n{confidence*100:.1f}% confidence", fontsize=11)
        plt.show()

        print("Full class probabilities:")
        for cname, p in sorted(zip(class_names, all_preds), key=lambda x: -x[1]):
            print(f"  {cname:35s} {p*100:5.1f}%")

uploader.observe(on_upload_change, names="value")
display(uploader, output)


### Notes

- If confidence is consistently low or predictions look wrong on real photographed leaves
  (as opposed to PlantVillage-style images), that is the **domain gap** your proposal
  discusses — expected, and worth reporting honestly with example images in your paper.
- Potato has only 3 classes in PlantVillage (Early blight, Late blight, healthy) — expect
  higher accuracy than a wider multi-crop model, since there's less inter-class confusion.
- The next stage of the project (not in this notebook) converts the saved model into a
  quantized TFLite/TFLite Micro file sized for your target hardware.
